In [1]:
import sys; sys.path.insert(0, "..")
from common import *

import time
import cv2
import pandas as pd
import tifffile
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, ZoeDepthForDepthEstimation

MODEL_ID = "Intel/zoedepth-nyu-kitti"   # trained on NYU (indoor) and KITTI (outdoor)
USE_FLIP = True                         # average with the flipped image, as in the original paper
STREAM = "B"
SPLIT = "dev"                           # do NOT change to "test" before 30 October

out_dir = stream_results_dir(CFG, STREAM)
seq_dir = sequence_dir(CFG, SPLIT)
print("Device:", DEVICE, "| Sequence:", seq_dir.name)

W1024 17:11:00.924000 40056 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Device: mps | Sequence: trans_t1_a


In [2]:
def load_frame(color_path):
    rgb = cv2.cvtColor(cv2.imread(str(color_path)), cv2.COLOR_BGR2RGB)
    depth_path = color_path.with_name(color_path.name.replace("_color.png", "_depth.tiff"))
    raw = tifffile.imread(depth_path)
    gt_mm = raw.astype(np.float32) / 65535.0 * 100.0
    valid = (raw > 0) & (raw < 65535) & (rgb.max(axis=2) > 10)
    return rgb, gt_mm, valid


def compute_metrics(pred, gt, valid):
    p = np.clip(pred[valid], 1e-3, None)
    g = gt[valid]
    absrel = np.mean(np.abs(p - g) / g)
    rmse = np.sqrt(np.mean((p - g) ** 2))
    delta1 = np.mean(np.maximum(p / g, g / p) < 1.25)
    return absrel, rmse, delta1


def scale_none(pred_m, gt, valid):
    return pred_m * 1000.0, 1000.0                       # metres to mm


def scale_median(pred_m, gt, valid):
    s = np.median(gt[valid]) / np.median(pred_m[valid])
    return pred_m * s, s


def scale_lsq_depth(pred_m, gt, valid):
    A = np.stack([pred_m[valid], np.ones(valid.sum())], axis=1)
    (a, b), *_ = np.linalg.lstsq(A, gt[valid], rcond=None)
    return pred_m * a + b, a


SCALINGS = {
    "none (metric x1000)": scale_none,
    "median": scale_median,
    "lsq_depth": scale_lsq_depth,
}

color_paths = sorted(seq_dir.glob("*_color.png"))
print(len(color_paths), "frames")

61 frames


In [3]:
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = ZoeDepthForDepthEstimation.from_pretrained(MODEL_ID).to(DEVICE).eval()


def predict_zoe(rgb):
    image = Image.fromarray(rgb)
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inputs)
        out_flip = model(pixel_values=torch.flip(inputs.pixel_values, dims=[3])) if USE_FLIP else None
    post = processor.post_process_depth_estimation(
        out,
        source_sizes=[(rgb.shape[0], rgb.shape[1])],
        outputs_flipped=out_flip,
    )
    return post[0]["predicted_depth"].float().cpu().numpy()   # metres, size of the input image

preprocessor_config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.38GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/647 [00:00<?, ?it/s]

In [4]:
rows, preds, times = [], {}, []

for cp in tqdm(color_paths):
    rgb, gt, valid = load_frame(cp)

    sync(DEVICE); t0 = time.perf_counter()
    pred_m = predict_zoe(rgb)
    sync(DEVICE); times.append(time.perf_counter() - t0)

    preds[cp.stem] = pred_m.astype(np.float16)

    for name, fn in SCALINGS.items():
        pred_mm, s = fn(pred_m, gt, valid)
        absrel, rmse, d1 = compute_metrics(pred_mm, gt, valid)
        rows.append({"frame": cp.stem, "scaling": name, "scale": s,
                     "absrel": absrel, "rmse": rmse, "delta1": d1})

per_frame = pd.DataFrame(rows)
print(f"Mean time per frame: {np.mean(times[1:]):.2f} s (flip = {USE_FLIP})")

100%|██████████| 61/61 [01:25<00:00,  1.40s/it]

Mean time per frame: 1.26 s (flip = True)


In [5]:
summary = (per_frame.groupby("scaling")[["absrel", "rmse", "delta1"]]
           .mean().round(3).reset_index())
summary.insert(0, "method", "ZoeDepth NYU+KITTI")
print(summary.to_string(index=False))

# How far is the metric scale from the colon scale?
med = per_frame[per_frame.scaling == "median"]["scale"]
print(f"\nMedian scale factor: {med.median():.1f} (1000 = metric output is correct in mm)")

tag = f"zoedepth_{SPLIT}"
per_frame.to_csv(out_dir / f"{tag}_per_frame.csv", index=False)
summary.to_csv(out_dir / f"{tag}_summary.csv", index=False)
np.savez_compressed(out_dir / f"{tag}_preds.npz", **preds)      # not in GitHub (.gitignore)
save_run_info(out_dir, DEVICE, SEED, extra={
    "model": MODEL_ID, "flip_test": USE_FLIP, "split": SPLIT,
    "sec_per_frame": float(np.mean(times[1:])),
})
print("Saved to", out_dir)

            method             scaling  absrel     rmse  delta1
ZoeDepth NYU+KITTI           lsq_depth   0.423    6.601   0.451
ZoeDepth NYU+KITTI              median   0.575   16.137   0.279
ZoeDepth NYU+KITTI none (metric x1000)  85.573 1283.060   0.000

Median scale factor: 14.2 (1000 = metric output is correct in mm)
Saved to /Users/aditiraghu/c3vd/results/B


In [6]:
from IPython.display import display

# Names for the table
SCALING_LABELS = {
    "median": "Median scaling",
    "lsq_depth": "LSQ (depth)",
    "none (metric x1000)": "No scaling (metric)",
}
COLUMNS = {
    "method": "Model",
    "scaling": "Scaling",
    "absrel": "AbsRel ↓",
    "rmse": "RMSE (mm) ↓",
    "delta1": "δ<1.25 ↑",
}


def show_table(df, title):
    styled = (
        df.rename(columns=COLUMNS)
          .style
          .format({"AbsRel ↓": "{:.3f}", "RMSE (mm) ↓": "{:.2f}", "δ<1.25 ↑": "{:.3f}"})
          .highlight_min(subset=["AbsRel ↓", "RMSE (mm) ↓"], props="font-weight: bold")
          .highlight_max(subset=["δ<1.25 ↑"], props="font-weight: bold")
          .hide(axis="index")
          .set_caption(title)
          .set_table_styles([
              {"selector": "caption", "props": "font-size: 14px; font-weight: bold; text-align: left; padding-bottom: 6px;"},
              {"selector": "th", "props": "text-align: left; padding: 4px 12px;"},
              {"selector": "td", "props": "padding: 4px 12px;"},
          ])
    )
    display(styled)


# 1. ZoeDepth only
zoe = summary.copy()
zoe["scaling"] = pd.Categorical(zoe["scaling"], categories=list(SCALING_LABELS), ordered=True)
zoe = zoe.sort_values("scaling")
zoe["scaling"] = zoe["scaling"].map(SCALING_LABELS)
show_table(zoe, f"ZoeDepth (NYU+KITTI) on {seq_dir.name}, {len(color_paths)} frames")

# 2. Comparison with the earlier dev results
# Numbers from the first notebook (dev set). Replace with CSV files when streams A and B save them.
reference = pd.DataFrame([
    ["Constant median (baseline)", "None",             0.603, 18.31, 0.256],
    ["DAv2 Small",                 "Median scaling",   0.352,  8.29, 0.528],
    ["DAv2 Small",                 "LSQ (disparity)",  0.312, 14.09, 0.440],
    ["DAv2 Base",                  "Median scaling",   0.375,  7.45, 0.524],
    ["DAv2 Base",                  "LSQ (disparity)",  0.321, 13.31, 0.418],
], columns=["method", "scaling", "absrel", "rmse", "delta1"])

zoe_rows = zoe[zoe["scaling"] != "No scaling (metric)"].copy()
zoe_rows["method"] = "ZoeDepth NYU+KITTI"
comparison = pd.concat([reference, zoe_rows[["method", "scaling", "absrel", "rmse", "delta1"]]],
                       ignore_index=True)
show_table(comparison, f"Model comparison on {seq_dir.name} (dev)")

# 3. Scale factor in plain words
factor = 1000.0 / med.median()
print(f"ZoeDepth predicts the colon about {factor:.0f} times farther away than it is "
      f"(median scale factor {med.median():.1f}; 1000 would mean correct metric scale).")

comparison.to_csv(out_dir / "model_comparison_dev.csv", index=False)

Model,Scaling,AbsRel ↓,RMSE (mm) ↓,δ<1.25 ↑
ZoeDepth NYU+KITTI,Median scaling,0.575,16.14,0.279
ZoeDepth NYU+KITTI,LSQ (depth),0.423,6.60,0.451
ZoeDepth NYU+KITTI,No scaling (metric),85.573,1283.06,0.000


Model,Scaling,AbsRel ↓,RMSE (mm) ↓,δ<1.25 ↑
Constant median (baseline),None,0.603,18.31,0.256
DAv2 Small,Median scaling,0.352,8.29,0.528
DAv2 Small,LSQ (disparity),0.312,14.09,0.440
DAv2 Base,Median scaling,0.375,7.45,0.524
DAv2 Base,LSQ (disparity),0.321,13.31,0.418
ZoeDepth NYU+KITTI,Median scaling,0.575,16.14,0.279
ZoeDepth NYU+KITTI,LSQ (depth),0.423,6.60,0.451


ZoeDepth predicts the colon about 70 times farther away than it is (median scale factor 14.2; 1000 would mean correct metric scale).
